In [ ]:
!pip install dytr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.6 MB/s eta 0:00:00


In [ ]:
from dytr import DynamicTransformer, PretrainedModelLoader, ModelConfig, TaskConfig, TrainingStrategy, Trainer, SingleDatasetProcessing
import pandas as pd

#model_name='prajjwal1/bert-tiny'
model_name='asafaya/bert-mini-arabic'


# 1. Configure your transformer
config = ModelConfig(
    embed_dim=256,#this will be changed automatically for finetune model
    num_layers=6,#
    num_heads=8,#
    max_seq_len=256,
    tokenizer_name = model_name,
    use_simple_tokenizer =True,
    special_tokens ={},
    per_device_train_batch_size=64,
    num_train_epochs=5,
    per_device_eval_batch_size=16,

)

#model loading

In [ ]:
# pretrained model

loader = PretrainedModelLoader()

## to confirm the model name is supported or not before loading,
info = loader.get_model_info(model_name)
print(f"\nModel info: {info}")

## model loading
model=loader.load_pretrained(model_name,config)

# to build model from scratch use the following line instead
#model=DynamicTransformer(config)


Model info: {'model_name': 'asafaya/bert-mini-arabic', 'model_type': 'bert', 'supported': True, 'architecture': {'hidden_size': 256, 'num_layers': 4, 'num_heads': 4, 'vocab_size': 32000, 'max_position_embeddings': 512}}

Loading BERT model: asafaya/bert-mini-arabic


config.json: 100%|██████████| 509/509 [00:00<00:00, 2.93MB/s]


pytorch_model.bin: 100%|██████████| 46.6M/46.6M [00:00<00:00, 233MB/s]


vocab.txt: 334kB [00:00, 29.1MB/s]


BERT config:
  Hidden size: 256
  Layers: 4
  Attention heads: 4
  Max position embeddings: 512
  Vocabulary size: 32000

Initializing DynamicTransformer...


************************************************************


Tokenizer loaded with vocab size: 32000

Loading model weights...
  Mapped word embeddings: shape torch.Size([32000, 256])
  Mapped layer 0
  Mapped layer 1
  Mapped layer 2
  Mapped layer 3
  Initialized final layer norm with identity

✓ Successfully loaded BERT model as encoder
  Encoder parameters: 12,404,224
  Total model parameters: 12,404,224
  Embed dim: 256
  Layers: 4
  Heads: 4
  Vocabulary size: 32000

📝 Note: This model has no Tasks: Add tasks using model.add_task() or Train the model on different Tasks to be added


In [ ]:

# congrats the pretrained model now became dytr architecture
model

DynamicTransformer(
  (shared_embedding): Embedding(32000, 256, padding_idx=0)
  (encoder): TransformerEncoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (attention): MultiHeadAttention(
          (q_proj): Linear(in_features=256, out_features=256, bias=True)
          (k_proj): Linear(in_features=256, out_features=256, bias=True)
          (v_proj): Linear(in_features=256, out_features=256, bias=True)
          (out_proj): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (attention_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (gate_proj): Linear(in_features=256, out_features=1024, bias=True)
          (up_proj): Linear(in_features=256, out_features=1024, bias=True)
          (down_proj): Linear(in_features=1024,

In [ ]:
tokenizer =model.tokenizer
print(tokenizer.encode(" السلام عليكم"))

{'input_ids': [2675, 4240], 'attention_mask': [1, 1]}


#load your dataset

In [ ]:
from datasets import load_dataset
dataset_tr = load_dataset('alsubari/arabic-grammar-errors', split='validation')
dataset_tr=pd.DataFrame(dataset_tr)
dataset_tr.head()

,text,correct_text,label,tags
0,مما يؤكل منه قال لهما ( ما أصبتما من أخيكما أع...,مما يؤكل منه قال لهما ( ما أصبتما من أخيكما أع...,3,0 0 0 0 0 0 0 0 0 0 0 0 237 0 0 36 0 0 0 0 0 0...
1,وتسعى السلطنة لتشجيع وتطوير هذا القطاع ليتيح ف...,وتسعى السلطنة لتشجيع وتطوير هذا القطاع ليتيح ف...,3,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 95 0 0 0 0 0 0 0...
2,خلال اقتراح بعض الحلول المناسبة للمشكلات والصع...,خلال اقتراح بعض الحلول المناسبة للمشكلات والصع...,6,0 0 0 0 0 0 0 0 0 0 0 0 115 0 0 0 0 0 0 0 0 0 ...
3,في الحديث الصحيح مع النبي عليه وعلى آله وصحبه ...,في الحديث الصحيح عن النبي عليه وعلى آله وصحبه ...,6,0 0 0 105 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ...
4,كيلومترات تكون بداية الانطلاقة من أمام مستشفى ...,كيلومترات تكون بداية الانطلاقة من أمام مستشفى ...,6,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ...


In [ ]:

# we make the tags labels as binary either 0,1 to avoid complexity and test the model
dataset_tr['tags'] = dataset_tr['tags'].str.split().apply(lambda tags: ' '.join('1' if int(t) != 0 else '0' for t in tags))
dataset_tr.head()

,text,correct_text,label,tags
0,مما يؤكل منه قال لهما ( ما أصبتما من أخيكما أع...,مما يؤكل منه قال لهما ( ما أصبتما من أخيكما أع...,3,0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 ...
1,وتسعى السلطنة لتشجيع وتطوير هذا القطاع ليتيح ف...,وتسعى السلطنة لتشجيع وتطوير هذا القطاع ليتيح ف...,3,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 ...
2,خلال اقتراح بعض الحلول المناسبة للمشكلات والصع...,خلال اقتراح بعض الحلول المناسبة للمشكلات والصع...,6,0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 ...
3,في الحديث الصحيح مع النبي عليه وعلى آله وصحبه ...,في الحديث الصحيح عن النبي عليه وعلى آله وصحبه ...,6,0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ...
4,كيلومترات تكون بداية الانطلاقة من أمام مستشفى ...,كيلومترات تكون بداية الانطلاقة من أمام مستشفى ...,6,0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 ...


## Data processing

In [ ]:

from sklearn.model_selection import train_test_split

dataset_tr, dataset_val = train_test_split(dataset_tr, test_size=0.1, random_state=42,stratify=dataset_tr['label'])

### Toke classification data

In [ ]:
#Token Classificatiin Data
train_token = SingleDatasetProcessing(
                dataset_tr,
                tokenizer,
                256,# max_length
                "error_detection",# task_name
                 TrainingStrategy.TOKEN_CLASSIFICATION,
                 text_column='text',
                 tags_column='tags',
                 token_labeling_first_only=False)
val_token = SingleDatasetProcessing(dataset_val, tokenizer, 256,"error_detection", TrainingStrategy.TOKEN_CLASSIFICATION,text_column='text', tags_column='tags',token_labeling_first_only=False,label_to_ids=train_token.label_to_ids)
error_detection_config=TaskConfig(
    task_name="error_detection",
    training_strategy=TrainingStrategy.TOKEN_CLASSIFICATION,
    datasets=[],
    num_labels=train_token.num_labels,
    max_length=256
    )

    Sample text: متطلعا ثم البحر والغمام الذي يعانقه ويختلط لونه مع لونه تقف و ( الكوس ) الباردة تهب عليك ....
    Sample tags: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...
  Created 14590 samples for error_detection >> token_classification
    Sample text: الله صلى الله عليه وسلم حتى مات، ومع آبيين بكر رضي الله عنه حتى مات، ومع عمر رضي الله عنه، فنحن نغزو عنك، فأبى فجهزوه فركب البحر فمات، فلم يجدوا ....
    Sample tags: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...
  Created 1622 samples for error_detection >> token_classification


## Multi dataset
lets add different datasets with different strategies

In [ ]:
# lets add different data for other tasks

## generation data with Question Answering
from dytr import process_qa_dataset

dataset_tr_qa=load_dataset('FreedomIntelligence/Code-Alpaca-Arabic-GPT4',split='train')

df_qa=process_qa_dataset(dataset_tr_qa,model.config,conversations_col="conversations")
train_df_qa, val_df_qa = train_test_split(df_qa, test_size=0.01, random_state=42)
train_causal = SingleDatasetProcessing(
        train_df_qa, tokenizer, 128,
        "generation", TrainingStrategy.CAUSAL_LM,
        text_column='text',#cache_dir=os.path.join(exp_dir,"dataset_cache"),
    )

val_causal = SingleDatasetProcessing(
        val_df_qa, tokenizer, 128,
        "generation", TrainingStrategy.CAUSAL_LM,
        text_column='text',#cache_dir=os.path.join(exp_dir,"dataset_cache"),
    )
generation_config=TaskConfig(
         task_name="generation",
         training_strategy=TrainingStrategy.CAUSAL_LM,
         max_length=128,
        )

## aother Ner tasks data

dataset_val_ner=load_dataset("iSemantics/conllpp-ner-ar",split='test')
dataset_tr_ner=load_dataset("iSemantics/conllpp-ner-ar",split='train')
dataset_val_ner=pd.DataFrame(dataset_val_ner)
dataset_tr_ner=pd.DataFrame(dataset_tr_ner)
dataset_tr_ner['ner_tags'] = dataset_tr_ner['ner_tags'].apply(lambda x: ' '.join(map(str,x)) if isinstance(x, list) else '')
dataset_tr_ner['tokens'] = dataset_tr_ner['tokens'].apply(lambda x: ' '.join(map(str,x)) if isinstance(x, list) else '')
dataset_val_ner['ner_tags'] = dataset_val_ner['ner_tags'].apply(lambda x: ' '.join(map(str,x)) if isinstance(x, list) else '')
dataset_val_ner['tokens'] = dataset_val_ner['tokens'].apply(lambda x: ' '.join(map(str,x)) if isinstance(x, list) else '')

train_token_ner = SingleDatasetProcessing(
        dataset_tr_ner, tokenizer, 256,
        "ner_detection", TrainingStrategy.TOKEN_CLASSIFICATION,
        text_column='tokens', tags_column='ner_tags',
    )
val_token_ner = SingleDatasetProcessing(
        dataset_val_ner, tokenizer, 256,
        "ner_detection", TrainingStrategy.TOKEN_CLASSIFICATION,
        text_column='tokens', tags_column='ner_tags',label_to_ids=train_token_ner.label_to_ids
    )
ner_detection_config=TaskConfig(
            task_name="ner_detection",
            training_strategy=TrainingStrategy.TOKEN_CLASSIFICATION,
            datasets=[],max_length=256,
            num_labels=train_token_ner.num_labels,
        )
## semantics data , sentence classification
dataset_tr_sem=load_dataset('sepidmnorozy/Arabic_sentiment',split='train')
dataset_val_sem=load_dataset('sepidmnorozy/Arabic_sentiment',split='test')
dataset_tr_sem=pd.DataFrame(dataset_tr_sem)
dataset_val_sem  =pd.DataFrame(dataset_val_sem)
train_class_sm = SingleDatasetProcessing(
        dataset_tr_sem, tokenizer, 128,
        "sentiment", TrainingStrategy.SENTENCE_CLASSIFICATION,
        text_column='text', label_column='label'
    )
val_class_sm = SingleDatasetProcessing(
        dataset_val_sem, tokenizer, 128,
        "sentiment", TrainingStrategy.SENTENCE_CLASSIFICATION,
        text_column='text', label_column='label')
sentiment_config=TaskConfig(
         task_name="sentiment",
         training_strategy=TrainingStrategy.SENTENCE_CLASSIFICATION,
         datasets=[],max_length=128,
         num_labels=train_class_sm.num_labels,
        )

  Sample processed text: Q: أعد الفهرس لأول حدوث لعنصر ما في المصفوفة المعطاة.
arr = [2, 3, 3, 1, 5, 2] 
 <|answer|> 
 إذا كنت تبحث عن فهرس أول ظهور لعنصر معين في القائمة، يمكنك استخدام الأمر index في ال Python. 

مثلا، إذا كنت تبحث عن أول وقوع للرقم 3، يمكنك القيام بما يلي:

index = arr.index(3)
يمكنك استبدل الرقم 3 بأي رقم (أو عنصر) تبحث عنه.

لذا القطعة البرمجية الكاملة ستكون كما يلي:

arr = [2, 3, 3, 1, 5, 2]
index = arr.index(3)
print(index)

عند تشغيل هذا، سيكون الناتج 1 (لأن العنصر 3 يظهر أولاً في الموقع 1 قائمه ال arr). 
 
...
Total Questions Answers: 20017
  Creating causal LM dataset with dynamic window sampling...


Processing documents: 100%|██████████| 19816/19816 [00:12<00:00, 1584.28it/s]


Do you want to save Causal Data Window in cach dir?[Y/n]
    Created and cached 74944 windows from 19816 documents
  Created 74944 samples for generation >> causal_lm
  Creating causal LM dataset with dynamic window sampling...


Processing documents: 100%|██████████| 201/201 [00:00<00:00, 1613.72it/s]


Do you want to save Causal Data Window in cach dir?[Y/n]
    Created and cached 812 windows from 201 documents
  Created 812 samples for generation >> causal_lm
    Sample text: الاتحاد الأوروبي يرفض الدعوة الألمانية لمقاطعة لحم الضأن البريطاني ....
    Sample tags: [3, 4, 0, 0, 7, 0, 0, 0, 7, 0]...
  Created 8926 samples for ner_detection >> token_classification
    Sample text: كرة القدم - اليابان تحقق فوزًا محظوظًا، الصين في هزيمة مفاجئة....
    Sample tags: [0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0]...
  Created 2202 samples for ner_detection >> token_classification
    Sample text: ربك دايما جنبك لو نديته هتلاقيه ...
    Sample label: 1
  Created 2468 samples for sentiment >> sentence_classification
    Sample text: سلامة أحمد سلامة، تركت إرثاً من الأخلاق حين عزت و من الحصافة حين اهتزت و من الشجاعة حين كان لها ثمن يرحمك الله...
    Sample label: 1
  Created 706 samples for sentiment >> sentence_classification


In [ ]:
train_datasets = {
        "sentiment": (train_class_sm, TrainingStrategy.SENTENCE_CLASSIFICATION),
        "error_detection": (train_token, TrainingStrategy.TOKEN_CLASSIFICATION),
        "ner_detection": (train_token_ner, TrainingStrategy.TOKEN_CLASSIFICATION),
        "generation": (train_causal, TrainingStrategy.CAUSAL_LM)
    }

val_datasets = {
        "sentiment": (val_class_sm, TrainingStrategy.SENTENCE_CLASSIFICATION),
        "error_detection": (val_token, TrainingStrategy.TOKEN_CLASSIFICATION),
        "ner_detection": (val_token_ner, TrainingStrategy.TOKEN_CLASSIFICATION),
        "generation": (val_causal, TrainingStrategy.CAUSAL_LM)
    }
task_configs_list=[sentiment_config,error_detection_config,ner_detection_config, generation_config]

# Training on multiple Tasks simultaneously

In [ ]:
model.config

ModelConfig(embed_dim=256, num_layers=4, num_heads=4, head_dim=64, ff_mult=4, dropout=0.1, max_seq_len=512, learning_rate=0.0003, batch_size=16, weight_decay=0.01, gradient_clip=1.0, warmup_steps=1000, max_learning_rate=0.0005, min_learning_rate=1e-06, adam_epsilon=1e-08, label_smoothing=0.1, fp16=False, gradient_accumulation_steps=1, max_grad_norm=1.0, patience=3, evaluation_strategy='steps', logging_steps=50, validation_check_interval=500, load_best_model_at_end=True, metric_for_best_model='loss', early_stopping_patience=10, max_train_steps=100000, num_train_epochs=5, lr_scheduler_type='cosine', per_device_train_batch_size=64, per_device_eval_batch_size=16, dataloader_num_workers=2, dataloader_pin_memory=True, seed=42, task_specific_lr={}, task_weights={}, use_rotary_embedding=False, use_flash_attention=False, gradient_checkpointing=False, training_from_scratch=False, special_tokens={}, window_size=256, stride=64, tasks={}, vocab_size=32000, tokenizer_name='asafaya/bert-mini-arabic',

In [ ]:
Train_config=model.config
Train_config.evaluation_strategy='epoch'
Train_config.num_train_epochs=10
Train_config.head_lr_mult=1.0
Train_config.decoder_lr_mult=2.0 # increased for generation which based on decoder
Train_config.shared_lr_mult=0.2
Train_config.learning_rate=3e-4
Train_config.per_device_train_batch_size=128
Train_config.per_device_eval_batch_size=32

In [ ]:

exp_dir='dytr_model'
trainer = Trainer(model, Train_config, exp_dir)
model=trainer.train(task_configs_list, train_datasets, val_datasets)

2026-04-02 14:52:47 - dytr.training.trainer - INFO - Trainer initialized
2026-04-02 14:52:47 - dytr.training.trainer - INFO - ============================================================
2026-04-02 14:52:47 - dytr.training.trainer - INFO - Starting training session
2026-04-02 14:52:47 - dytr.training.trainer - INFO - ============================================================
2026-04-02 14:52:47 - dytr.training.trainer - INFO -   Frozen all model parameters
2026-04-02 14:52:47 - dytr.training.trainer - INFO -   Unfrozen shared encoder and embeddings
2026-04-02 14:52:47 - dytr.training.trainer - INFO -   Unfrozen head for task: sentiment
2026-04-02 14:52:47 - dytr.training.trainer - INFO -   Unfrozen head for task: error_detection
2026-04-02 14:52:47 - dytr.training.trainer - INFO -   Unfrozen head for task: ner_detection
2026-04-02 14:52:47 - dytr.training.trainer - INFO -   Unfrozen decoder for task: generation
2026-04-02 14:52:47 - dytr.training.trainer - INFO - Model parameters aft

Max lengths per task: {'sentiment': 128, 'error_detection': 256, 'ner_detection': 256, 'generation': 128}
Max lengths per task: {'sentiment': 128, 'error_detection': 256, 'ner_detection': 256, 'generation': 128}


Epoch 1/10: 100%|██████████| 786/786 [05:24<00:00,  2.42it/s, loss=4.5472, tasks={'generation': 4.1654456424713135, 'error_detection': 0.35143900513648985, 'ner_detection': 1.3261612837805468, 'sentiment': 0.6868754344827989}]



  Epoch 1 average training loss: 4.4872


2026-04-02 14:58:14 - dytr.training.trainer - WARNING -   ✓ Best model saved improvement: inf


...


  Validation loss: 1.0990
    sentiment validation:
      accuracy: 0.7855
      f1_macro: 0.7798
      f1_weighted: 0.7860
      f1_micro: 0.7855
    error_detection validation:
      token_accuracy: 0.9302
      f1_macro: 0.5974
      f1_weighted: 0.9175
      f1_micro: 0.9302
    ner_detection validation:
      token_accuracy: 0.8000
      f1_macro: 0.3578
      f1_weighted: 0.7853
      f1_micro: 0.8000
    generation validation:
      perplexity: 24.6720
      token_accuracy: 0.6076
  Early stopping triggered after 9 Validation
Insert letter Y to stop the training: [Y/n]


Epoch 10/10: 100%|██████████| 786/786 [05:22<00:00,  2.43it/s, loss=2.0433, tasks={'generation': 2.624660816192627, 'error_detection': 0.23840540289878845, 'ner_detection': 0.585954545175328, 'sentiment': 0.2000701935852275}]



  Epoch 10 average training loss: 2.0459


2026-04-02 15:48:13 - dytr.training.trainer - WARNING -   No improvement for 1 epochs



  Validation loss: 1.0993
    sentiment validation:
      accuracy: 0.7855
      f1_macro: 0.7798
      f1_weighted: 0.7860
      f1_micro: 0.7855
    error_detection validation:
      token_accuracy: 0.9304
      f1_macro: 0.5969
      f1_weighted: 0.9176
      f1_micro: 0.9304
    ner_detection validation:
      token_accuracy: 0.7994
      f1_macro: 0.3567
      f1_weighted: 0.7850
      f1_micro: 0.7994
    generation validation:
      perplexity: 24.6604
      token_accuracy: 0.6083


In [ ]:

#model_name='asafaya/bert-mini-arabic'
model.save_model("finetune_bert_mini_arabic_mltitasks_and_generation.pt")

In [ ]:
model.eval()

DynamicTransformer(
  (shared_embedding): Embedding(32000, 256, padding_idx=0)
  (encoder): TransformerEncoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (attention): MultiHeadAttention(
          (q_proj): Linear(in_features=256, out_features=256, bias=True)
          (k_proj): Linear(in_features=256, out_features=256, bias=True)
          (v_proj): Linear(in_features=256, out_features=256, bias=True)
          (out_proj): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (attention_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (gate_proj): Linear(in_features=256, out_features=1024, bias=True)
          (up_proj): Linear(in_features=256, out_features=1024, bias=True)
          (down_proj): Linear(in_features=1024,

# model inference

In [ ]:



inp="q: اشرح ما هو الخوارزمية غير القاطعة. \n <|answer|> \n"
model.generate(inp,task_name ="generation")

'q : اشرح ما هو الخوارزمية غير القاطعة . < | a n s w e r | > عدد ال و ل في القا م ة ( ط و ل ) في البرمجة هي 0 ، 1 ) حيث كل عنصر في القا م ة يبد من الصفر حتى تصل ل ى البداية . " 2 .'

In [ ]:
inp='الله صلى الله عليه وسلم حتى مات، ومع آبيين بكر رضي الله عنه حتى مات، ومع عمر رضي الله عنه، فنحن نغزو عنك، فأبى فجهزوه فركب البحر فمات، فلم يجدوا .'
print("error_detection ")
model.generate(inp,task_name ="error_detection")["pairs"]

error_detection 


[('الله', 0),
 ('صلى', 0),
 ('الله', 0),
 ('عليه', 0),
 ('وسلم', 0),
 ('حتى', 0),
 ('مات', 0),
 ('،', 0),
 ('ومع', 0),
 ('[UNK]', 0),
 ('ب', 0),
 ('ي', 1),
 ('ي', 1),
 ('ن', 1),
 ('بكر', 0),
 ('رضي', 0),
 ('الله', 0),
 ('عنه', 0),
 ('حتى', 0),
 ('مات', 0),
 ('،', 0),
 ('ومع', 0),
 ('عمر', 0),
 ('رضي', 0),
 ('الله', 0),
 ('عنه', 0),
 ('،', 0),
 ('فنحن', 0),
 ('نغ', 0),
 ('##زو', 0),
 ('عنك', 0),
 ('،', 0),
 ('ف', 0),
 ('[UNK]', 0),
 ('ب', 0),
 ('ى', 0),
 ('فج', 0),
 ('##هز', 0),
 ('##وه', 0),
 ('فرك', 0),
 ('##ب', 0),
 ('البحر', 0),
 ('فمات', 0),
 ('،', 0),
 ('فلم', 0),
 ('يجدوا', 0),
 ('.', 0)]

In [ ]:

inp= " قامة شركة ستارلنك بعمل حملة توعوية بقيادة ايلون ماسك"
model.generate(inp,task_name ="ner_detection")["pairs"]

[('قام', 0),
 ('##ة', 0),
 ('شركة', 0),
 ('ستار', 3),
 ('##لن', 4),
 ('##ك', 0),
 ('بعمل', 0),
 ('حملة', 0),
 ('تو', 0),
 ('##عو', 0),
 ('##ية', 0),
 ('بقيادة', 0),
 ('ايل', 0),
 ('##ون', 0),
 ('ماسك', 0)]

In [ ]:
model.generate(inp,task_name ="sentiment")

{'prediction': 0,
 'probabilities': [0.9577341675758362, 0.04226585477590561],
 'logits': [1.5720747709274292, -1.548515796661377]}

# Model Export

In [ ]:
exporter = model.get_exporter()
sentiment_model = exporter.export_single_task("sentiment", "./sentiment_model.pt")

In [ ]:
sentiment_model

SingleTaskModel(
  (encoder): TransformerEncoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (layers): ModuleList(
      (0-3): 4 x EncoderLayer(
        (attention): MultiHeadAttention(
          (q_proj): Linear(in_features=256, out_features=256, bias=True)
          (k_proj): Linear(in_features=256, out_features=256, bias=True)
          (v_proj): Linear(in_features=256, out_features=256, bias=True)
          (out_proj): Linear(in_features=256, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (attention_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (ffn): FeedForward(
          (gate_proj): Linear(in_features=256, out_features=1024, bias=True)
          (up_proj): Linear(in_features=256, out_features=1024, bias=True)
          (down_proj): Linear(in_features=1024, out_features=256, bias=True)
          (activation): GELU(app

In [ ]:
print(f"   Total model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Total parameters of sentiment_model: {sum(p.numel() for p in sentiment_model.parameters()):,}")

   Total model parameters: 25,009,293
   Total parameters of sentiment_model: 12,437,634


#continue training
here we fine tune the model on generation data or you can add new data with new task configuration, and we put on validation three different tasks include generation,

In [ ]:

model=DynamicTransformer.load_model("finetune_bert_mini_arabic_mltitasks_and_generation.pt")

train_datasets = {

        "generation": (train_causal, TrainingStrategy.CAUSAL_LM)
    }

val_datasets = {

        "error_detection": (val_token, TrainingStrategy.TOKEN_CLASSIFICATION),
        "ner_detection": (val_token_ner, TrainingStrategy.TOKEN_CLASSIFICATION),
        "generation": (val_causal, TrainingStrategy.CAUSAL_LM)
    }
task_configs_list=[sentiment_config,error_detection_config,ner_detection_config, generation_config]
#task_configs_list=[ generation_config]
Train_config=model.config
Train_config.per_device_train_batch_size=64
Train_config.num_train_epochs=5
Train_config.head_lr_mult=0.05
Train_config.decoder_lr_mult=1.0 # increased for generation which based on decoder
Train_config.shared_lr_mult=0.05
Train_config.learning_rate=3e-4
exp_dir='dytr_model_continue_training'
model.best_val_loss=10
trainer = Trainer(model, Train_config, exp_dir)
model=trainer.train(task_configs_list, train_datasets, val_datasets)

File already exists: downloads/bert_mini_arabic/vocab.txt
************************************************************
File already exists: downloads/bert_mini_arabic/special_tokens_map.json


2026-04-02 17:11:05 - dytr.training.trainer - INFO - Trainer initialized
2026-04-02 17:11:05 - dytr.training.trainer - INFO - ============================================================
2026-04-02 17:11:05 - dytr.training.trainer - INFO - Starting training session
2026-04-02 17:11:05 - dytr.training.trainer - INFO - ============================================================
2026-04-02 17:11:05 - dytr.training.trainer - INFO -   Frozen all model parameters
2026-04-02 17:11:05 - dytr.training.trainer - INFO -   Unfrozen shared encoder and embeddings
2026-04-02 17:11:05 - dytr.training.trainer - INFO -   Unfrozen decoder for task: generation
2026-04-02 17:11:05 - dytr.training.trainer - INFO - Model parameters after adding tasks:
2026-04-02 17:11:05 - dytr.training.trainer - INFO -   Model size: 25009293 parameters
2026-04-02 17:11:05 - dytr.training.trainer - INFO -   Trainable: 24840448
2026-04-02 17:11:05 - dytr.training.trainer - INFO - 
Parameter breakdown:
2026-04-02 17:11:05 - d

Max lengths per task: {'sentiment': 128, 'error_detection': 256, 'ner_detection': 256, 'generation': 128}
Max lengths per task: {'sentiment': 128, 'error_detection': 256, 'ner_detection': 256, 'generation': 128}


Epoch 1/5: 100%|██████████| 1171/1171 [05:00<00:00,  3.89it/s, loss=2.6929, tasks={'generation': 2.7645734977722167}]



  Epoch 1 average training loss: 2.6947

  Validation loss: 1.1850
    error_detection validation:
      token_accuracy: 0.9309
      f1_macro: 0.5951
      f1_weighted: 0.9176
      f1_micro: 0.9309
    ner_detection validation:
      token_accuracy: 0.7995
      f1_macro: 0.3581
      f1_weighted: 0.7847
      f1_micro: 0.7995
    generation validation:
      perplexity: 25.8841
      token_accuracy: 0.5956


...


  Validation loss: 1.1798
    error_detection validation:
      token_accuracy: 0.9311
      f1_macro: 0.5853
      f1_weighted: 0.9165
      f1_micro: 0.9311
    ner_detection validation:
      token_accuracy: 0.8000
      f1_macro: 0.3563
      f1_weighted: 0.7834
      f1_micro: 0.8000
    generation validation:
      perplexity: 25.5279
      token_accuracy: 0.6105


Epoch 5/5: 100%|██████████| 1171/1171 [04:59<00:00,  3.90it/s, loss=2.5115, tasks={'generation': 2.4971592569351198}]



  Epoch 5 average training loss: 2.5116


2026-04-02 17:36:20 - dytr.training.trainer - WARNING -   ✓ Best model saved improvement: 0.0001728649322803033



  Validation loss: 1.1796
    error_detection validation:
      token_accuracy: 0.9312
      f1_macro: 0.5851
      f1_weighted: 0.9165
      f1_micro: 0.9312
    ner_detection validation:
      token_accuracy: 0.8003
      f1_macro: 0.3572
      f1_weighted: 0.7838
      f1_micro: 0.8003
    generation validation:
      perplexity: 25.5176
      token_accuracy: 0.6103
